# NeMo Fraud Detection – Datenpipeline erklärt

Dieses Notebook erklärt die drei bereitgestellten Skripte des Projekts `nemo-fraud-detection`:

1. **Noise Injection Pipeline** – erzeugt gezielt verrauschte Daten.
2. **NeMo Curator Pipeline** – bereinigt, anonymisiert, filtert und dedupliziert die Daten.
3. **SFT Data Preparation** – verbindet kuratierte Texte mit Ground-Truth-Labels und erstellt Train/Validation/Test.

## Gesamtablauf

```text
transcripts.jsonl
      │
      ▼
Noise Injection
      │
      ▼
transcripts_noisy.jsonl
      │
      ▼
NeMo Curator
 ├─ Unicode-Reformatierung
 ├─ PII-Maskierung
 ├─ Mindestlängenfilter
 └─ Exact Deduplication
      │
      ▼
transcripts_curated.jsonl
      │
      ├─────────────── transcripts_benchmark.jsonl
      │                         │
      └──────── Matching ───────┘
                │
                ▼
          70 / 15 / 15 Split
          │       │       │
        train     val    test


# 1. Noise Injection Pipeline

Die erste Pipeline nimmt rohe JSONL-Transkripte und injiziert künstliche Störungen.

| Fehlertyp | Wahrscheinlichkeit |
|---|---:|
| Exakte Duplikate | 8 % |
| PII | 7 % |
| Garbage | 5 % |
| Encoding-Fehler | 3 % |
| sauber | ca. 77 % |

Die Bedingungen sind kumulativ:

```text
0.00–0.07  Duplicate
0.08–0.14  PII
0.15–0.19  Garbage
0.20–0.22  Encoding
0.23–0.99  Clean
```

Die konkrete Verteilung ist wegen `random.random()` von Lauf zu Lauf unterschiedlich.

In [1]:
import sys
import json
import random
from pathlib import Path
from typing import Any, Dict, List

BASE_DIR = Path("/data")
DATA_DIR = BASE_DIR / "nemo-fraud-detection" / "notebooks" / "01_Data_Generation" / "data"

ORIGINAL_RAW_PATH = DATA_DIR / "transcripts.jsonl"
NOISY_RAW_PATH = DATA_DIR / "transcripts_noisy.jsonl"

NOISY_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)

PROB_DUPLICATE = 0.08
PROB_PII = 0.07
PROB_GARBAGE = 0.05
PROB_ENCODING = 0.03


## 1.1 Pfade und Konfiguration

`Path` kapselt die Dateipfade. `mkdir(..., exist_ok=True)` stellt sicher, dass das Zielverzeichnis existiert.

Die vier `PROB_*`-Variablen steuern die Noise-Injektion. Der verbleibende Wahrscheinlichkeitsbereich bleibt unverändert.

In [2]:
PII_SAMPLES = [
    "Meine Kreditkartennummer lautet 4532-8921-1029-4411 mit CVV 892.",
    "Sie erreichen mich unter max.mustermann@example.de oder Mobil: +49 171 1234567.",
    "Meine IBAN lautet DE89 3704 0044 0532 0130 00, Inhaber ist Thomas Müller.",
    "Ich wohne in der Hauptstraße 45, 10115 Berlin. Geburtsdatum ist der 14.05.1982.",
    "Meine Sozialversicherungsnummer ist 12 140582 M 043."
]

GARBAGE_SNIPPETS = [
    "???", "N/A", "asdfghjkl;", "NULL",
    "[ERROR_VOICEMAIL_RECORDING_CORRUPTED]",
    "CLICK... BEEP... BEEP...", "a"
]


## 1.2 PII- und Garbage-Beispiele

`PII_SAMPLES` simuliert sensible Daten. `GARBAGE_SNIPPETS` simuliert beschädigte oder inhaltlich wertlose Textfragmente.

Diese Beispiele sind später wichtig, weil die Curator-Pipeline solche Probleme wieder erkennen bzw. entschärfen soll.

In [4]:
def load_jsonl(file_path: Path) -> List[Dict[str, Any]]:
    records = []
    if not file_path.exists():
        return records

    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line_str = line.strip()
            if not line_str:
                continue
            try:
                records.append(json.loads(line_str))
            except json.JSONDecodeError as e:
                print(f"⚠️ Warnung: Ungültiges JSON in Zeile {line_no}: {e}")
    return records


def format_to_text(item: Dict[str, Any]) -> str:
    if "text" in item and isinstance(item["text"], str):
        return item["text"]

    raw_response = item.get("response", "")
    if not raw_response:
        return ""

    try:
        parsed = json.loads(raw_response) if isinstance(raw_response, str) else raw_response
        if isinstance(parsed, dict) and "transcript" in parsed:
            return "\n".join(
                f"{t.get('speaker', 'Unbekannt')}: {t.get('text', '')}"
                for t in parsed["transcript"]
            )
    except (json.JSONDecodeError, TypeError):
        pass

    return ""


## 1.3 JSONL laden und Text extrahieren

`load_jsonl()` verarbeitet die Datei zeilenweise. Ein ungültiger JSON-Eintrag erzeugt eine Warnung.

`format_to_text()` unterstützt zwei Strukturen:

- ein direktes Feld `text`
- ein `response`-Feld mit einer `transcript`-Liste

Dadurch kann das Skript unterschiedliche Rohformate vereinheitlichen.

In [5]:
def main():
    print(f"📖 Lade Original-Transkripte von: {ORIGINAL_RAW_PATH}")
    input_records = load_jsonl(ORIGINAL_RAW_PATH)

    if not input_records:
        print(
            f"❌ FEHLER: Keine Daten in '{ORIGINAL_RAW_PATH}' "
            "gefunden oder Datei existiert nicht! Abbruch."
        )
        sys.exit(1)

    print(
        f"🧬 Injiziere Datenmüll, PII, Duplikate & Encoding-Fehler "
        f"in {len(input_records)} Einträge..."
    )

    noisy_dataset = []
    stats = {"duplicate": 0, "pii": 0, "garbage": 0, "encoding": 0, "clean": 0}

    for idx, item in enumerate(input_records):
        call_id = item.get("call_id", item.get("id", f"CALL_{idx:04d}"))
        flat_text = format_to_text(item)

        if not flat_text:
            continue

        dice = random.random()

        if dice < PROB_DUPLICATE:
            noisy_dataset.append({
                "call_id": call_id,
                "text": flat_text,
                "is_clean": True,
                "noise_type": "none"
            })
            noisy_dataset.append({
                "call_id": f"{call_id}_DUP",
                "text": flat_text,
                "is_clean": False,
                "noise_type": "exact_duplicate"
            })
            stats["duplicate"] += 1

        elif dice < (PROB_DUPLICATE + PROB_PII):
            pii_text = f"{flat_text}\nAnrufer: {random.choice(PII_SAMPLES)}"
            noisy_dataset.append({
                "call_id": call_id,
                "text": pii_text,
                "is_clean": False,
                "noise_type": "pii_injection"
            })
            stats["pii"] += 1

        elif dice < (PROB_DUPLICATE + PROB_PII + PROB_GARBAGE):
            noisy_dataset.append({
                "call_id": call_id,
                "text": random.choice(GARBAGE_SNIPPETS),
                "is_clean": False,
                "noise_type": "garbage_text"
            })
            stats["garbage"] += 1

        elif dice < (
            PROB_DUPLICATE + PROB_PII + PROB_GARBAGE + PROB_ENCODING
        ):
            broken_text = (
                flat_text
                .replace("ä", "Ã¤")
                .replace("ö", "Ã¶")
                .replace("ü", "Ã¼")
                .replace("ß", "Ã\x9f")
            )
            noisy_dataset.append({
                "call_id": call_id,
                "text": broken_text,
                "is_clean": False,
                "noise_type": "encoding_error"
            })
            stats["encoding"] += 1

        else:
            noisy_dataset.append({
                "call_id": call_id,
                "text": flat_text,
                "is_clean": True,
                "noise_type": "none"
            })
            stats["clean"] += 1

    print(f"💾 Schreibe verunreinigte Testdaten nach: {NOISY_RAW_PATH}")
    with open(NOISY_RAW_PATH, "w", encoding="utf-8") as f:
        for rec in noisy_dataset:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print("\n📊 INJEKTIONS-STATISTIK:")
    print(f"  • Original: {len(input_records)}")
    print(f"  • Duplikate: {stats['duplicate']}")
    print(f"  • PII: {stats['pii']}")
    print(f"  • Garbage: {stats['garbage']}")
    print(f"  • Encoding: {stats['encoding']}")
    print(f"  • Clean: {stats['clean']}")
    print(f"\n✅ {len(noisy_dataset)} Einträge erstellt.")


if __name__ == "__main__":
    main()


📖 Lade Original-Transkripte von: /data/nemo-fraud-detection/notebooks/01_Data_Generation/data/transcripts.jsonl
🧬 Injiziere Datenmüll, PII, Duplikate & Encoding-Fehler in 154 Einträge...
💾 Schreibe verunreinigte Testdaten nach: /data/nemo-fraud-detection/notebooks/01_Data_Generation/data/transcripts_noisy.jsonl

📊 INJEKTIONS-STATISTIK:
  • Original: 154
  • Duplikate: 15
  • PII: 11
  • Garbage: 1
  • Encoding: 4
  • Clean: 123

✅ 169 Einträge erstellt.


### 1.4 Zentrale Noise-Logik

Für jeden Datensatz wird eine Zufallszahl erzeugt. Je nach Bereich wird eine Störung ausgewählt.

**Besonders wichtig:** Beim Duplicate-Fall werden zwei Einträge geschrieben: das Original und eine Kopie mit `_DUP`. Deshalb kann die Ausgabedatei größer als die Eingabedatei werden.

# 2. NeMo Curator Pipeline

Diese Pipeline bereinigt `transcripts_noisy.jsonl`.

```text
Schema-Validierung
       ↓
NeMo Curator Dataset
       ↓
ID hinzufügen
       ↓
Unicode-Reformatierung
       ↓
PII-Maskierung
       ↓
MinLengthFilter
       ↓
ExactDuplicates / MD5
       ↓
transcripts_curated.jsonl
```

Das Ziel ist ein bereinigter und anonymisierter Datensatz.

In [6]:
import sys
import json
import re
import shutil
from pathlib import Path
from typing import Any
import os

from nemo_curator import Modify, ScoreFilter, Sequential, AddId
from nemo_curator.datasets import DocumentDataset
from nemo_curator.filters import DocumentFilter
from nemo_curator.modules import ExactDuplicates
from nemo_curator.utils.distributed_utils import get_client
from nemo_curator.modifiers import DocumentModifier, UnicodeReformatter

from dask_cuda import LocalCUDACluster
from distributed import Client


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-22 13:34:25.966530930 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


## 2.1 GPU-/CUDA-Infrastruktur

NeMo Curator wird mit einem GPU-/Dask-Backend initialisiert. Die Imports `LocalCUDACluster` und `Client` sind im bereitgestellten Skript vorhanden; die eigentliche `main()`-Funktion verwendet sie jedoch nicht direkt.

Die drei gesetzten Environment Variables dienen laut Quellcode als Workaround für cuFile/GDS-Probleme in Container-Umgebungen.

In [8]:
from pathlib import Path
import os

BASE_DIR = Path("/data")
DATA_DIR = BASE_DIR / "nemo-fraud-detection" / "notebooks" 

INPUT_PATH = DATA_DIR / "01_Data_Generation" / "data" / "transcripts_noisy.jsonl"
CURATED_OUT_PATH = DATA_DIR / "02_Data_Curation" / "data" / "transcripts_curated.jsonl"
TEMP_EXPORT_DIR = DATA_DIR / "02_Data_Curation" / "data"  / "_temp_curator_export"
DEDUP_LOG_DIR = DATA_DIR / "02_Data_Curation" / "data"  / "dedup_logs"
DEDUP_CACHE_DIR = DATA_DIR / "02_Data_Curation" / "data"  / "dedup_cache"

# Verzeichnisse automatisch erstellen lassen
CURATED_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
DEDUP_LOG_DIR.mkdir(parents=True, exist_ok=True)
DEDUP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Environment Variables als Workaround für cuFile/GDS-Probleme in Containern
os.environ["KVIKIO_COMPAT_MODE"] = "ON"
os.environ["CUDF_CUFILE_ENABLED"] = "0"
os.environ["RAPIDS_NO_CUFILE"] = "1"

print("✅ Pfade und CUDA-Umgebung erfolgreich initialisiert!")
print(f"Erwartete Eingabedatei: {INPUT_PATH}")


✅ Pfade und CUDA-Umgebung erfolgreich initialisiert!
Erwartete Eingabedatei: /data/nemo-fraud-detection/notebooks/01_Data_Generation/data/transcripts_noisy.jsonl


## 2.2 Strenge Eingabevalidierung

Die Funktion `validate_input_file()` verwendet das Fail-Fast-Prinzip.

Abbruch erfolgt, wenn:

- die Datei fehlt,
- JSON ungültig ist,
- `text` fehlt,
- die Datei leer ist.

Dadurch werden Fehler früh erkannt, bevor die GPU-/Curator-Pipeline startet.

In [9]:
def validate_input_file(file_path: Path) -> int:
    print(f"🔍 Validiere Eingabedatei & Schema: {file_path.name}...")

    if not file_path.exists():
        print(f"❌ KRITISCHER FEHLER: '{file_path}' existiert nicht!")
        sys.exit(1)

    total_lines = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line_str = line.strip()
            if not line_str:
                continue

            try:
                record = json.loads(line_str)
            except json.JSONDecodeError as e:
                print(
                    f"❌ SCHEMAFEHLER [Zeile {line_no}]: "
                    f"Kein valides JSON. Fehler: {e}"
                )
                sys.exit(1)

            if "text" not in record:
                print(
                    f"❌ SCHEMAFEHLER [Zeile {line_no}]: "
                    "Pflichtfeld 'text' fehlt!"
                )
                sys.exit(1)

            total_lines += 1

    if total_lines == 0:
        print("❌ KRITISCHER FEHLER: Eingabedatei ist vollkommen leer!")
        sys.exit(1)

    print(f"✅ Validation erfolgreich! {total_lines} Einträge.")
    return total_lines


## 2.3 PII-Maskierung

`FraudPiiModifier` ist eine eigene NeMo-Curator-Erweiterung. Sie verwendet reguläre Ausdrücke, um bestimmte Muster zu ersetzen:

- IBAN
- Kreditkartennummern
- E-Mail-Adressen
- Geburtsdaten
- Handynummern
- Adressen
- Kundennummern

Beispielsweise wird eine erkannte IBAN durch `[IBAN_MASKIERT]` ersetzt.

Die Erkennung ist dabei **musterbasiert**: Nur Schreibweisen, die zum jeweiligen Regex passen, werden erkannt.

In [10]:
class FraudPiiModifier(DocumentModifier):
    def modify_document(self, doc: str) -> str:
        if not isinstance(doc, str):
            return doc

        doc = re.sub(
            r'DE\d{2}\s?(\d{4}\s?){4}\d{2}',
            '[IBAN_MASKIERT]', doc
        )

        doc = re.sub(
            r'\b(?:\d[ -]*?){13,16}\b',
            '[KREDITKARTE_MASKIERT]', doc
        )

        doc = re.sub(
            r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
            '[EMAIL_MASKIERT]', doc
        )

        doc = re.sub(
            r'\b\d{1,2}\.\s+'
            r'(?:Januar|Februar|März|April|Mai|Juni|Juli|August|'
            r'September|Oktober|November|Dezember)\s+\d{4}\b',
            '[GEBURTSDATUM_MASKIERT]', doc, flags=re.IGNORECASE
        )

        doc = re.sub(
            r'\b(?:\+49|0)\s*1[567]\d[\s\-]?\d{3,8}\b',
            '[HANDYNUMMER_MASKIERT]', doc
        )

        doc = re.sub(
            r'\b(?:[A-ZÄÖÜ][a-zäöüß]+\s+)?'
            r'[A-ZÄÖÜ][a-zäöüß]+'
            r'(?:straße|str\.|weg|allee|platz|ring)\s+\d+[a-zA-Z]?,?'
            r'\s*\d{5}\s+[A-ZÄÖÜ][a-zäöüß]+\b',
            '[ADRESSE_MASKIERT]', doc, flags=re.IGNORECASE
        )

        doc = re.sub(
            r'\b\d{9}\b',
            '[KUNDENNUMMER_MASKIERT]', doc
        )

        return doc


## 2.4 Mindestlängenfilter

`MinLengthFilter` berechnet die Länge des Textes.

```python
return len(str(text).strip())
```

`keep_document()` gibt nur dann `True` zurück, wenn mindestens 50 Zeichen vorhanden sind.

Der Filter kann dabei ein Dictionary, ein Objekt mit Text-Attribut oder einen String verarbeiten.

In [11]:
class MinLengthFilter(DocumentFilter):
    def __init__(self, min_length: int = 50, text_field: str = "text"):
        super().__init__()
        self.min_length = min_length
        self.text_field = text_field

    def score_document(self, doc: Any) -> int:
        if isinstance(doc, dict):
            text = doc.get(self.text_field, "")
        elif hasattr(doc, self.text_field):
            text = getattr(doc, self.text_field, "")
        elif isinstance(doc, str):
            text = doc
        else:
            return 0

        return len(str(text).strip())

    def keep_document(self, score: int) -> bool:
        return score >= self.min_length


In [13]:
import os
os.environ["DASK_DATAFRAME__QUERY_PLANNING"] = "False"
def main():
    initial_count = validate_input_file(INPUT_PATH)

    print("🚀 Initialisiere NeMo Curator Execution Client (GPU/CUDA Backend)...")
    client = get_client(cluster_type="gpu", set_torch_to_use_rmm=False)
    print("🔗 Dask-Client erfolgreich verbunden.")

    print("⚡ 1. Lade Dataset in NeMo Curator...")
    dataset = DocumentDataset.read_json(str(INPUT_PATH), add_filename=True, backend="pandas")

    print("🆔 2. Generiere eindeutige IDs für NeMo Curator (Voraussetzung für Deduplizierung)...")
    add_id = AddId(id_field="id", id_prefix="FRAUD_data", start_index=0)
    dataset = add_id(dataset)

    print("🛡️ 3. Wende Cleaning-Sequenz an (Unicode-Reformat via ftfy & PII-Maskierung)...")
    cleaners = Sequential([
        Modify(UnicodeReformatter()),
        Modify(FraudPiiModifier())
    ])
    dataset = cleaners(dataset).persist()

    print("🧹 4. Wende NeMo Curator MinLengthFilter an (Min 50 Zeichen)...")
    length_filter = ScoreFilter(
        MinLengthFilter(min_length=50, text_field="text"),
        score_type=int
    )
    dataset = length_filter(dataset)

    print("✂️ 5. Führe Exakte Deduplizierung (ExactDuplicates) aus...")
    exact_dup = ExactDuplicates(
        logger=str(DEDUP_LOG_DIR),
        id_field="id",
        text_field="text",
        hash_method="md5",
        cache_dir=str(DEDUP_CACHE_DIR),
    )
    duplicates_dataset = exact_dup(dataset=dataset)
    
    # Identifiziere Duplikate, die entfernt werden sollen
    exact_docs_to_remove = duplicates_dataset.df.map_partitions(
        lambda x: x[x._hashes.duplicated(keep="first")]
    )

    # Filter herausfiltern
    id_field = "id"
    cleaned_df = dataset.df[
        ~dataset.df[id_field].isin(exact_docs_to_remove[id_field].compute())
    ]
    dataset = DocumentDataset(cleaned_df)

    # Aufräumen alter Temp-Ordner und alter Ziel-Dateien
    if TEMP_EXPORT_DIR.exists():
        shutil.rmtree(TEMP_EXPORT_DIR)
    if CURATED_OUT_PATH.exists():
        if CURATED_OUT_PATH.is_dir():
            shutil.rmtree(CURATED_OUT_PATH)
        else:
        
            CURATED_OUT_PATH.unlink()

    print(f"💾 Schreibe finalen, kurierten Datensatz nach: {CURATED_OUT_PATH}")
    dataset.to_json(str(TEMP_EXPORT_DIR), write_to_filename=False)

    # Zusammenführen der Partitions-Dateien
    exported_files = list(TEMP_EXPORT_DIR.glob("*.json*")) + list(TEMP_EXPORT_DIR.glob("*.part"))
    if exported_files:
        shutil.move(str(exported_files[0]), str(CURATED_OUT_PATH))
        shutil.rmtree(TEMP_EXPORT_DIR)
    else:
        with open(CURATED_OUT_PATH, "w", encoding="utf-8") as outfile:
            for part in sorted(TEMP_EXPORT_DIR.iterdir()):
                if part.is_file():
                    with open(part, "r", encoding="utf-8") as infile:
                        shutil.copyfileobj(infile, outfile)
        shutil.rmtree(TEMP_EXPORT_DIR)

    # Finale Auswertung
    final_docs = []
    with open(CURATED_OUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                final_docs.append(json.loads(line))
    
    final_count = len(final_docs)
    total_removed = initial_count - final_count

    print("\n" + "="*80)
    print("📊 NEMO CURATOR ADVANCED PIPELINE STATISTIK")
    print("="*80)
    print(f"  • Eingelesene Roh-Datensätze:      {initial_count}")
    print(f"  • Gefilterte / Duplizierte Einträge: {total_removed}")
    print("--------------------------------------------------------------------------------")
    print(f"  • Verbliebene kurierte Datensätze: {final_count}")
    print(f"  • Verwerfungsquote Total:          {(total_removed / initial_count):.2%}")
    print("="*80)
    print(f"\n✅ Pipeline erfolgreich beendet! Saubere Datei: {CURATED_OUT_PATH.name}")

if __name__ == "__main__":
    main()

🔍 Validiere Eingabedatei & Schema: transcripts_noisy.jsonl...
✅ Validation erfolgreich! 169 Einträge.
🚀 Initialisiere NeMo Curator Execution Client (GPU/CUDA Backend)...


/usr/local/lib/python3.10/dist-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37705 instead
  warnings.warn(
2026-08-22 13:45:30.067834959 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


🔗 Dask-Client erfolgreich verbunden.
⚡ 1. Lade Dataset in NeMo Curator...
Reading 1 files
🆔 2. Generiere eindeutige IDs für NeMo Curator (Voraussetzung für Deduplizierung)...
🛡️ 3. Wende Cleaning-Sequenz an (Unicode-Reformat via ftfy & PII-Maskierung)...
🧹 4. Wende NeMo Curator MinLengthFilter an (Min 50 Zeichen)...
✂️ 5. Führe Exakte Deduplizierung (ExactDuplicates) aus...
💾 Schreibe finalen, kurierten Datensatz nach: /data/nemo-fraud-detection/notebooks/02_Data_Curation/data/transcripts_curated.jsonl
Writing to disk complete for 1 partitions

📊 NEMO CURATOR ADVANCED PIPELINE STATISTIK
  • Eingelesene Roh-Datensätze:      169
  • Gefilterte / Duplizierte Einträge: 16
--------------------------------------------------------------------------------
  • Verbliebene kurierte Datensätze: 153
  • Verwerfungsquote Total:          9.47%

✅ Pipeline erfolgreich beendet! Saubere Datei: transcripts_curated.jsonl


# 3. SFT Data Preparation

Die dritte Pipeline verbindet kuratierte Texte mit Ground-Truth-Labels.

```text
transcripts_curated.jsonl
          │
          │ ID
          ▼
       Matching  ◄──── transcripts_benchmark.jsonl
          │
          ▼
{"input": Text, "output": Label}
          │
          ▼
Shuffle mit Seed 42
          │
          ▼
70 % / 15 % / 15 %
          │
          ├── train.jsonl
          ├── validation.jsonl
          └── test.jsonl
```

In [17]:
import sys
import json
import random
from pathlib import Path

BASE_DIR = Path("/data")
DATA_DIR = BASE_DIR / "nemo-fraud-detection" / "notebooks"

CURATED_PATH = DATA_DIR / "02_Data_Curation" / "data" / "transcripts_curated.jsonl"
BENCHMARK_PATH = DATA_DIR / "01_Data_Generation" / "data"/ "benchmark.jsonl"
SFT_DIR = DATA_DIR / "02_Data_Curation" / "data" / "sft"


## 3.1 Fail-Fast und JSONL-Helfer

Das Skript prüft zuerst, ob beide Eingabedateien vorhanden sind.

`load_jsonl()` liest Datensätze ein.

`save_jsonl()` schreibt sie wieder zeilenweise als JSONL.

In [18]:
def validate_required_files():
    print("🔍 Prüfe Eingabedateien...")

    if not CURATED_PATH.exists():
        print(f"❌ Kuratierte Datei nicht gefunden: '{CURATED_PATH}'")
        sys.exit(1)

    if not BENCHMARK_PATH.exists():
        print(f"❌ Benchmark-Datei nicht gefunden: '{BENCHMARK_PATH}'")
        sys.exit(1)

    print("✅ Alle benötigten Dateien wurden gefunden.\n")


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


In [19]:
def main():
    validate_required_files()

    print("📊 Bereite SFT-Datensätze vor...")
    
    curated_data = load_jsonl(CURATED_PATH)
    benchmark_data = load_jsonl(BENCHMARK_PATH)

    # 1. Lookup-Map aus Benchmark aufbauen (Schlüssel: "id")
    gt_map = {}
    for item in benchmark_data:
        cid = item.get("id")
        label = item.get("label") or item.get("fraud_type")
        if cid and label:
            gt_map[str(cid)] = label

    # 2. Matching über 'call_id' oder 'id' ausführen
    sft_samples = []
    matched_count = 0
    unmatched_count = 0

    for doc in curated_data:
        text = doc.get("text", "")
        cid = str(doc.get("call_id") or doc.get("id", ""))
        
        gt_label = gt_map.get(cid)
        
        if gt_label:
            matched_count += 1
            sft_samples.append({
                "input": text,
                "output": gt_label
            })
        else:
            unmatched_count += 1

    total_docs = len(curated_data)
    print(f"ℹ️ Total geladene kuratierte Dokumente: {total_docs}")
    print(f"    🔗 Erfolgreich mit Ground-Truth gematcht: {matched_count}")
    
    if unmatched_count > 0:
        print(f"    ⚠️ Ohne passendes Label übersprungen: {unmatched_count} Dokument(e)")

    if len(sft_samples) == 0:
        print(f"\n❌ KRITISCHER FEHLER: Es konnten keine einzigen SFT-Daten gematcht werden!")
        sys.exit(1)

    # 3. Train / Val / Test Split (70% / 15% / 15%)
    random.seed(42)
    random.shuffle(sft_samples)

    total_samples = len(sft_samples)
    train_end = int(total_samples * 0.70)
    val_end = train_end + int(total_samples * 0.15)

    train_data = sft_samples[:train_end]
    val_data = sft_samples[train_end:val_end]
    test_data = sft_samples[val_end:]

    # 4. Speichern
    SFT_DIR.mkdir(parents=True, exist_ok=True)
    
    train_path = SFT_DIR / "train.jsonl"
    val_path = SFT_DIR / "validation.jsonl"
    test_path = SFT_DIR / "test.jsonl"

    save_jsonl(train_data, train_path)
    save_jsonl(val_data, val_path)
    save_jsonl(test_data, test_path)

    print(f"\n    -> Exportiert: train.jsonl ({len(train_data)} Einträge)")
    print(f"    -> Exportiert: validation.jsonl ({len(val_data)} Einträge)")
    print(f"    -> Exportiert: test.jsonl ({len(test_data)} Einträge)")
    print(f"\n✅ SFT-Datensätze erfolgreich in '{SFT_DIR}' gespeichert!")

if __name__ == "__main__":
    main()


🔍 Prüfe Eingabedateien...
✅ Alle benötigten Dateien wurden gefunden.

📊 Bereite SFT-Datensätze vor...
ℹ️ Total geladene kuratierte Dokumente: 153
    🔗 Erfolgreich mit Ground-Truth gematcht: 153

    -> Exportiert: train.jsonl (107 Einträge)
    -> Exportiert: validation.jsonl (22 Einträge)
    -> Exportiert: test.jsonl (24 Einträge)

✅ SFT-Datensätze erfolgreich in '/data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft' gespeichert!


# 4. Zusammenspiel der drei Skripte

Die drei Skripte bilden eine zusammenhängende Kette:

### Schritt 1 – Noise Injection

`transcripts.jsonl` → `transcripts_noisy.jsonl`

Es werden absichtlich Duplikate, PII, Garbage und Encoding-Probleme erzeugt.

### Schritt 2 – NeMo Curator

`transcripts_noisy.jsonl` → `transcripts_curated.jsonl`

Die Daten werden normalisiert, anonymisiert, gefiltert und dedupliziert.

### Schritt 3 – SFT Preparation

`transcripts_curated.jsonl` + `transcripts_benchmark.jsonl`

→ `train.jsonl`, `validation.jsonl`, `test.jsonl`

So entsteht aus Rohdaten ein strukturierter Input/Output-Datensatz für SFT.

# 5. Beispiel: Ein Datensatz durchläuft die Pipeline

### Rohdaten

```json
{"id": "doc-00001", "text": "Kunde: Meine Karte funktioniert nicht ..."}
```

### Noise Injection

Es könnte beispielsweise PII angehängt werden:

```text
Kunde: Meine Karte funktioniert nicht ...
Anrufer: Meine IBAN lautet ...
```

### Curator

Die IBAN kann anschließend zu

```text
[IBAN_MASKIERT]
```

werden.

Sehr kurze Texte oder exakte Duplikate können entfernt werden.

### SFT Preparation

Nach dem Matching entsteht:

```json
{
  "input": "Kunde: Meine Karte funktioniert nicht ...",
  "output": "legitimate"
}
```

Das ist das grundlegende SFT-Beispiel.

# 6. Wichtige Beobachtungen im bereitgestellten Code

## Noise Injection und Curator sind bewusst gekoppelt

Die erste Pipeline erzeugt unter anderem exakte Duplikate und Encoding-Probleme. Die zweite Pipeline soll diese anschließend bereinigen.

## IDs sind für das Matching entscheidend

Die SFT-Pipeline sucht über `call_id` oder `id`, während die Ground-Truth-Map über `id` aufgebaut wird. Stimmen diese IDs nicht überein, entstehen `unmatched` Datensätze.

## Die SFT-Pipeline prüft keine Klassenverteilung

Der bereitgestellte Code kontrolliert nicht separat, wie viele `fraud`- und `legitimate` Beispiele in Train, Validation und Test landen.

## Die Curator-Statistik fasst Entfernung zusammen

`total_removed` kombiniert Filterung und Deduplizierung. Es wird nicht getrennt ausgewiesen, welcher Mechanismus wie viele Dokumente entfernt hat.

## Ein Duplicate verändert die Anzahl

Bei einem Duplicate-Fall erzeugt die Noise-Pipeline zwei Datensätze aus einem Input. Deshalb darf man die Anzahl der Ausgabedatensätze nicht einfach mit der Eingabeanzahl gleichsetzen.

# 7. Lernfragen

1. Warum erzeugt die Noise-Pipeline absichtlich Duplikate?
2. Was passiert bei `dice = 0.18`?
3. Warum kann `transcripts_noisy.jsonl` größer sein als die Eingabe?
4. Welche PII-Typen versucht `FraudPiiModifier` zu erkennen?
5. Warum wird `AddId` verwendet?
6. Was bedeutet `duplicated(keep="first")`?
7. Warum wird `random.seed(42)` gesetzt?
8. Welche Aufgabe hat `transcripts_benchmark.jsonl`?
9. Was passiert mit einem kuratierten Dokument ohne Label?
10. Warum ist die Testmenge bei kleinen Datensätzen nicht zwingend exakt 15 %?

## Merksatz

**Noise erzeugt Probleme → Curator bereinigt Probleme → SFT Preparation verbindet Text und Label und teilt die Daten auf.**

# Fazit

Die drei Skripte implementieren zusammen eine vollständige Datenvorbereitung:

```text
Rohdaten
   ↓
Noise Injection
   ↓
Verrauschte Daten
   ↓
NeMo Curator
   ↓
Bereinigte / anonymisierte Daten
   ↓
Ground-Truth-Matching
   ↓
SFT Train / Validation / Test
```

Die bereitgestellte Pipeline ist damit klar in **Datenerzeugung/Robustheitstest**, **Datenkuration** und **SFT-Datenvorbereitung** getrennt.